## Import Dependency

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from nltk.corpus import stopwords
import nltk

In [2]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

In [3]:
import html
import torch
from torch import nn
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

C:\Users\Asus\Documents\Kuliah\Final-Thesis-Repository\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np

## Import Data 

In [5]:
file_path = "../data/output/data_for_model.csv"

In [6]:
df = pd.read_csv(file_path)
df

,full_text,clean_text,unit
0,"Password not recognised <p>Hello,&nbsp;</p>\n\...",password not recognised hello when i try to co...,DPTSI (Direktorat Pengembangan Teknologi dan S...
1,LUPA PASSWORD EMAILITS <p>Permisi Disini saya ...,lupa password emailits permisi disini saya ing...,DPTSI (Direktorat Pengembangan Teknologi dan S...
2,"Lupa pasword e-mail ITS <p>Selamat pagi, saya ...",lupa pasword e mail its selamat pagi saya yang...,DPTSI (Direktorat Pengembangan Teknologi dan S...
3,Tidak bisa login my ITS <p>Selamat pagi perken...,tidak bisa login my its selamat pagi perkenalk...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4,Tidak bisa login di myits <p>saya ingin merese...,tidak bisa login di myits saya ingin mereset p...,DPTSI (Direktorat Pengembangan Teknologi dan S...
...,...,...,...
4676,Akun myits tidak bisa terutentikasi <p>aplikas...,akun myits tidak bisa terutentikasi aplikasi a...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4677,"Tidak bisa login myits, karena apk authenticat...",tidak bisa login myits karena apk authenticato...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4678,Lupa password integra <p>Mohon bantuannya saya...,lupa password integra mohon bantuannya saya ma...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4679,Ganti Nomor Telpon MyIts <p>Nomor telpon myITS...,ganti nomor telpon myits nomor telpon myits sa...,DPTSI (Direktorat Pengembangan Teknologi dan S...


## Preprocess Before Modelling

Dropping the unknown label

In [7]:
print("Data before drop unknown : ", len(df))
df = df[df['unit'] != 'UNKNOWN']
print("Data after drop unknown : ", len(df))

Data before drop unknown :  4681
Data after drop unknown :  4633


## Stopwords Preparation

In [8]:
nltk.download('stopwords', quiet=True)
nltk_stopwords = nltk.corpus.stopwords.words('indonesian')
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

In [9]:
custom_stopwords = ['nya', 'sih', 'dong', 'min', 'mohon', 'bantuannya', 'terima', 'kasih', 'yg', 'utk', 'kak', 'pak', 'bu']

In [10]:
all_stopwords = list(set(nltk_stopwords + sastrawi_stopwords + custom_stopwords))

## SVM Modelling

In [11]:
df_svm = df.copy()

In [12]:
df_svm.unit.value_counts()

unit
DPTSI (Direktorat Pengembangan Teknologi dan Sistem Informasi)                 2961
DPSP (Direktorat Pendidikan Sarjana dan Pascasarjana)                           484
DSDMO (Direktorat Sumber Manusia dan Organisasi                                 314
UNCLASSIFIED                                                                    244
DITMAWA (Direktorat Kemahasiswaan)                                              241
UKP (Unit Komunikasi Publik)                                                    203
BK (Biro Keuangan)                                                               64
DPPS (Direktorat Perencanaan dan Pengembangan Strategis)                         47
DRPM (Direktorat Riset dan Pengabdian Masyarakat)                                30
BUK4L (Biro Umum dan Keamanan, Keselamatan, Kesehatan Kerja dan Lingkungan)      21
KPM (Kantor Penjaminan Mutu)                                                     13
ULH (Unit Layanan Hukum)                                               

In [13]:
X_svm = df_svm['clean_text']
y_svm = df_svm['unit']

In [14]:
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_svm, y_svm, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_svm 
)

In [15]:
y_train_svm.value_counts()

unit
DPTSI (Direktorat Pengembangan Teknologi dan Sistem Informasi)                 2369
DPSP (Direktorat Pendidikan Sarjana dan Pascasarjana)                           387
DSDMO (Direktorat Sumber Manusia dan Organisasi                                 251
UNCLASSIFIED                                                                    195
DITMAWA (Direktorat Kemahasiswaan)                                              193
UKP (Unit Komunikasi Publik)                                                    162
BK (Biro Keuangan)                                                               51
DPPS (Direktorat Perencanaan dan Pengembangan Strategis)                         38
DRPM (Direktorat Riset dan Pengabdian Masyarakat)                                24
BUK4L (Biro Umum dan Keamanan, Keselamatan, Kesehatan Kerja dan Lingkungan)      17
KPM (Kantor Penjaminan Mutu)                                                     10
ULH (Unit Layanan Hukum)                                               

In [16]:
y_test_svm.value_counts()

unit
DPTSI (Direktorat Pengembangan Teknologi dan Sistem Informasi)                 592
DPSP (Direktorat Pendidikan Sarjana dan Pascasarjana)                           97
DSDMO (Direktorat Sumber Manusia dan Organisasi                                 63
UNCLASSIFIED                                                                    49
DITMAWA (Direktorat Kemahasiswaan)                                              48
UKP (Unit Komunikasi Publik)                                                    41
BK (Biro Keuangan)                                                              13
DPPS (Direktorat Perencanaan dan Pengembangan Strategis)                         9
DRPM (Direktorat Riset dan Pengabdian Masyarakat)                                6
BUK4L (Biro Umum dan Keamanan, Keselamatan, Kesehatan Kerja dan Lingkungan)      4
KPM (Kantor Penjaminan Mutu)                                                     3
ULH (Unit Layanan Hukum)                                                         2

In [17]:
pipeline_svm = Pipeline([
    (
        'tfidf', 
        TfidfVectorizer(
            stop_words=all_stopwords,  # Menggunakan gabungan stopwords tadi
            max_features=5000,         # Membatasi 5000 kata paling penting untuk mencegah overfit
            ngram_range=(1, 2)         # Menggunakan unigram (1 kata) dan bigram (2 kata berurutan)
        )
    ),
    (
        'svm', 
        SVC(
            kernel='linear',           # Kernel Linear paling optimal untuk dimensi besar seperti teks
            class_weight='balanced',   # KUNCI UTAMA: Penyesuaian otomatis untuk kelas imbalanced
            random_state=42
        )
    )
])

In [18]:
pipeline_svm.fit(X_train_svm, y_train_svm)

C:\Users\Asus\Documents\Kuliah\Final-Thesis-Repository\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [19]:
y_pred_svm = pipeline_svm.predict(X_test_svm)

In [20]:
print(classification_report(y_test_svm, y_pred_svm, zero_division=0))

                                                                             precision    recall  f1-score   support

                                                         BK (Biro Keuangan)       0.83      0.77      0.80        13
BUK4L (Biro Umum dan Keamanan, Keselamatan, Kesehatan Kerja dan Lingkungan)       1.00      0.50      0.67         4
                                         DITMAWA (Direktorat Kemahasiswaan)       0.91      0.90      0.91        48
                   DPPS (Direktorat Perencanaan dan Pengembangan Strategis)       0.33      0.22      0.27         9
                      DPSP (Direktorat Pendidikan Sarjana dan Pascasarjana)       0.71      0.89      0.79        97
             DPTSI (Direktorat Pengembangan Teknologi dan Sistem Informasi)       0.97      0.91      0.94       592
                          DRPM (Direktorat Riset dan Pengabdian Masyarakat)       1.00      0.17      0.29         6
                            DSDMO (Direktorat Sumber Manusia da

## IndoBERT Modelling

In [21]:
df_indobert = df.copy()

In [22]:
def preprocess_for_bert(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text = text.lower().strip()
    return text # Stopwords & tanda baca TETAP DIPERTAHANKAN

In [23]:
df_indobert['preprocessed_text'] = df_indobert['full_text'].apply(preprocess_for_bert)

In [24]:
label_encoder = LabelEncoder()
df_indobert['label'] = label_encoder.fit_transform(df['unit'])
num_labels = len(label_encoder.classes_)

In [25]:
X_indobert = df_indobert['preprocessed_text'].tolist()
y_indobert = df_indobert['label'].tolist()

In [26]:
X_train_indobert, X_test_indobert, y_train_indobert, y_test_indobert = train_test_split(
    X_indobert, y_indobert, test_size=0.2, random_state=42, stratify=y_indobert
)

In [27]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [28]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

In [29]:
train_dataset = Dataset.from_dict({"text": X_train_indobert, "label": y_train_indobert})
test_dataset = Dataset.from_dict({"text": X_test_indobert, "label": y_test_indobert})

train_dataset = train_dataset.map(tokenize_function, batched=True).remove_columns(["text"])
test_dataset = test_dataset.map(tokenize_function, batched=True).remove_columns(["text"])

Map: 100%|██████████████████████████████████████████████████████████████████| 927/927 [00:00<00:00, 5923.50 examples/s]


In [30]:
class_weights = compute_class_weight('balanced', classes=np.arange(num_labels), y=y_train_indobert)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()

In [31]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Menggunakan CrossEntropyLoss dengan bobot kelas
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [32]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

You passed `num_labels=12` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 13126.99it/s]
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
# 1. Konfigurasi Training
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,              # Jumlah putaran belajar (3-5 biasanya optimal)
    per_device_train_batch_size=16,  # Jika memori GPU penuh (Out of Memory/OOM), turunkan jadi 8
    per_device_eval_batch_size=16,
    eval_strategy="epoch",           # Evaluasi dilakukan setiap akhir epoch
    learning_rate=2e-5,              
)

# 2. Inisiasi Custom Trainer (dengan bobot kelas yang sudah kita buat)
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# 3. Mulai Proses Fine-Tuning
print("Memulai proses fine-tuning IndoBERT...")
trainer.train()

Memulai proses fine-tuning IndoBERT...


C:\Users\Asus\Documents\Kuliah\Final-Thesis-Repository\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 